In [1]:
from graph_transformer_long_range_niches.tl import pad_batch
from graph_transformer_long_range_niches._paths import CFG_FILES, RESULTS, FIG_PATH, LEGNINI23
from graph_transformer_long_range_niches.model import LitGNNTransformer, LitGNNTransformerMasked
from graph_transformer_long_range_niches.modules import LitGCNMasked, LitGCN
from graph_transformer_long_range_niches.pl import CustomColormap, plot_attention_sender_receiver, plot_attention_sender_receiver, SelfAttentionRelevance, calculate_attention, normalized_attention, normalized_class_attention
from graph_transformer_long_range_niches.pp import prepare_geome_dataset, split_adata
from graph_transformer_long_range_niches.config import load_config
from graph_transformer_long_range_niches.tl.geome_dataloader import GraphAnnDataModule

import warnings
warnings.filterwarnings("ignore")

from torch_geometric.loader import DataLoader

from pathlib import Path
import wandb
import torch
import os

## plotting
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import pickle

import scanpy as sc
import squidpy as sq

from sklearn.preprocessing import MinMaxScaler

/ictstr01/groups/ml01/workspace/francesca.drummer/mamba/envs/GT_long_range_env/lib/python3.11/site-packages/numba/core/decorators.py:246: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


# InterScale training


0. Create `config.yaml` file
1. Load `cfg` and `adata` object
2. Prepare PyG data
3. Train GNN (Pre-training)
4. Train Transformer (Fine-tuning, global interactions)

In [2]:
#AXIS = 1 #gene
AXIS = 0 #cell
num_features = 2

In [3]:
y_true = torch.randn(5, num_features)
y_pred = torch.randn(5, num_features)

In [4]:
y_pred = y_pred.T.contiguous()
y_true = y_true.T.contiguous()

In [5]:
y_var = torch.var(y_true, dim=AXIS, keepdim=False)
y_var

tensor([1.1829, 0.0187, 0.1463, 2.4780, 2.3850])

In [6]:
import torchmetrics

r2_raw = torchmetrics.R2Score(num_outputs=num_features, multioutput = 'raw_values')
r2_raw.num_outputs

2

In [7]:
FOLDER = "legnini23"
cfg_path = "/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/src/config_files/Legnini_23/legnini23_InterScale_genes_sample.yaml"

### Load config and adata

In [8]:
cfg = load_config(cfg_path)

In [9]:
adata = sc.read_h5ad(cfg.dataset.h5ad_data)
adata

AnnData object with n_obs × n_vars = 43762 × 88
    obs: 'Cell', 'Area', 'x', 'y', 'sample', 'condition', 'organoid', 'obs_names'
    var: 'gene_ids', 'feature_types'
    obsm: 'spatial'
    layers: 'log1p_norm', 'norm_ftsqrt', 'raw'

In [10]:
adata.X = adata.layers['norm_ftsqrt']

In [11]:
# clustering

In [12]:
train_size, val_size, test_size = float(cfg.dataset.train_size), float(cfg.dataset.val_size), float(cfg.dataset.test_size)
print(f'Train size: {train_size}, val size: {val_size} and test size: {test_size}')

Train size: 0.7, val size: 0.2 and test size: 0.1


In [13]:
split_adata(adata, split_obs=cfg.dataset.obs_split, val_size=val_size, test_size=test_size, seed = cfg.optim.seed, stratify_groups = cfg.dataset.prediction_obs)

Test size > 0
{'counts': {'train': 23890, 'val': 13228, 'test': 6644}, 'groups': {'train': ['slide1_C2-1', 'slide1_C2-3', 'slide1_C2-5', 'slide4_A2-2', 'slide1_B2-3', 'slide1_A2-1', 'slide4_B2-3', 'slide4_A2-1', 'slide4_A2-3', 'slide1_A2-2', 'slide1_B2-2'], 'val': ['slide4_B2-2', 'slide1_C2-2', 'slide1_D2-3', 'slide1_D2-2'], 'test': ['slide4_B2-1', 'slide1_B2-1']}}


AnnData object with n_obs × n_vars = 43762 × 88
    obs: 'Cell', 'Area', 'x', 'y', 'sample', 'condition', 'organoid', 'obs_names', 'split'
    var: 'gene_ids', 'feature_types'
    obsm: 'spatial'
    layers: 'log1p_norm', 'norm_ftsqrt', 'raw'

In [14]:
# cross-gene vs cross-cell evaluation
cfg.set_new_allowed(True)
cfg.defrost()
cfg.optim.cross_corr = 'gene'
cfg.dataset.batch_size = 3
cfg.dataset.pct_mask_nodes = 0.3
#cfg.dataset.spatial_neigbors_kwargs.radius = 0
cfg.freeze()

In [15]:
pyg_data_list, _ = prepare_geome_dataset(adata, cfg)
pyg_data_list

call new
call new
call new
test
call new


[[Data(x=[3023, 88], edge_index=[2, 30734], obs_names=[3023]),
  Data(x=[1694, 88], edge_index=[2, 19178], obs_names=[1694]),
  Data(x=[1701, 88], edge_index=[2, 19342], obs_names=[1701]),
  Data(x=[2241, 88], edge_index=[2, 21678], obs_names=[2241]),
  Data(x=[2870, 88], edge_index=[2, 31620], obs_names=[2870]),
  Data(x=[1640, 88], edge_index=[2, 18170], obs_names=[1640]),
  Data(x=[1958, 88], edge_index=[2, 21830], obs_names=[1958]),
  Data(x=[2630, 88], edge_index=[2, 22200], obs_names=[2630]),
  Data(x=[2506, 88], edge_index=[2, 22690], obs_names=[2506]),
  Data(x=[1984, 88], edge_index=[2, 16736], obs_names=[1984]),
  Data(x=[1643, 88], edge_index=[2, 13926], obs_names=[1643])],
 [Data(x=[3510, 88], edge_index=[2, 40186], obs_names=[3510]),
  Data(x=[4318, 88], edge_index=[2, 48666], obs_names=[4318]),
  Data(x=[3621, 88], edge_index=[2, 39762], obs_names=[3621]),
  Data(x=[1779, 88], edge_index=[2, 13146], obs_names=[1779])],
 [Data(x=[3299, 88], edge_index=[2, 36290], obs_names

In [16]:
# y_pred = torch.randn(2, 4)  # Random values from normal distribution
# y_true = torch.randn(2, 4) 
# print(y_pred, y_true)
# n_cells = y_true.shape[0]
# print(y_pred.reshape(-1, n_cells), y_true.reshape(-1, n_cells))

# r2_gene = torchmetrics.R2Score(num_outputs=4, multioutput = 'raw_values')
# r2_cell = torchmetrics.R2Score(num_outputs=2, multioutput = 'raw_values')

# print(r2_gene(y_pred, y_true))
# print(r2_cell(y_pred.reshape(-1), y_true.reshape(-1)))

# mse = torchmetrics.MeanSquaredError()
# print(mse(y_pred, y_true))
# print(mse(y_pred.T.contiguous(), y_true.T.contiguous()))

In [17]:
train_ds, val_ds, test_ds = pyg_data_list[0], pyg_data_list[1], pyg_data_list[2]

In [18]:
# set number of classes and number of features
cfg.dataset.merge_from_list(['num_features', len(train_ds[0].x[1])])
if 'classification' in cfg.dataset.prediction_task:
  cfg.dataset.merge_from_list(['num_classes', len(train_ds[0].y[1])])

In [19]:
dm = GraphAnnDataModule(datas=pyg_data_list, 
                           num_workers=1, 
                           batch_size=int(cfg.dataset.batch_size), 
                           pct_mask_nodes=cfg.dataset.pct_mask_nodes,
                           learning_type="node")

Masked dataloader


In [22]:
#model = LitGCNMasked(cfg)
model = LitGNNTransformerMasked(cfg)

cross-gene per cellcorrelation metrics
cross-gene correlation metrics
num features: 88


In [23]:
import math
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor, EarlyStopping, Callback, Timer
import pytorch_lightning as pl

class MetricsHistory(Callback):
    def __init__(self):
        super().__init__()
        self.history = []
        
    def on_train_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        # Convert tensor values to float
        epoch_dict = {k: v.item() if hasattr(v, 'item') else v 
                     for k, v in metrics.items()}
        epoch_dict['epoch'] = trainer.current_epoch
        self.history.append(epoch_dict)

steps_per_epoch = math.ceil(len(train_ds) / cfg.dataset.batch_size)
early_stop_callback = EarlyStopping(monitor="val_r2", min_delta=0.05, patience=10*steps_per_epoch, verbose=False, mode="max")
lr_monitor = LearningRateMonitor(logging_interval='epoch')
timer = Timer()
history_callback = MetricsHistory()

In [24]:
trainer = pl.Trainer(min_epochs=1, 
                     max_epochs=10000,
                     enable_progress_bar=False,
                     callbacks=[lr_monitor, early_stop_callback, timer, history_callback],
                     log_every_n_steps=steps_per_epoch,
                     # Sanity checks: Debugging model
                     #overfit_batches=1,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [ ]:
trainer.fit(model, dm)
trainer.validate(model, dm)

You are using a CUDA device ('NVIDIA A100-PCIE-40GB MIG 3g.20gb') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-e479c0c7-6b16-58ac-a77b-40ea9ff345cd]

  | Name                   | Type                       | Params | Mode 
------------------------------------------------------------------------------
0 | loss                   | GaussianNLLLoss            | 0      | train
1 | norm_input             | LayerNorm                  | 32     | train
2 | gnn                    | LitGCN                     | 15.0 K | train
3 | transformer_encoder    | TransformerNodeEncoderHook | 19.3 K | train
4 | graph_pred_linear_list | ModuleList                 | 0      | train
5 | graph_pred_linear      

In [ ]:
# After training, get the time metrics
total_time = timer.time_elapsed("train")  # total training time in seconds
avg_epoch_time = timer.time_elapsed("train") / trainer.current_epoch  # average time per epoch

print(f"Total training time: {total_time:.2f} seconds")
print(f"Average epoch time: {avg_epoch_time:.2f} seconds")

In [ ]:
def plot_history(history_callback, subset_term=None):
    history = pd.DataFrame(history_callback.history)
    if subset_term is not None:
        history = history[[c for c in history.columns if subset_term in c]]
    ax = history[[c for c in history.columns if 'train' in c]].plot()
    plt.gca().set_prop_cycle(None)
    history[[c for c in history.columns if 'val' in c]].plot(style='--', ax=ax)
    plt.grid(True)
    plt.xlabel('Epoch')
    plt.legend()
    plt.show()

plot_history(history_callback, subset_term='mse')
plot_history(history_callback, subset_term='r2')

### Test dataloader

In [73]:
dm.setup("fit")

In [74]:
for i, batch in enumerate(dm.train_dataloader()):
    print(batch, np.unique(batch.batch))
    _, mask_idx = apply_mask(batch)

DataBatch(x=[3627, 88], edge_index=[2, 30662], obs_names=[3627], mask=[3627], batch=[3627], ptr=[3]) [0 1]


NameError: name 'apply_mask' is not defined

## Model evaluation

In [66]:
# samples in test set 
adata[adata.obs['split'] == 'test'].obs.groupby(['split', 'sample', 'condition']).size()

split  sample       condition
test   slide1_B2-1  SHH          3299
       slide4_B2-1  SHH          3345
dtype: int64

In [67]:
dm.setup('test')

### GNNTransformer

In [51]:
for i, batch in enumerate(dm.test_dataloader()):
    print(batch)
    padded_h_node, transformer_out, src_padding_mask, index_nodes, dec_out = model.evaluation(batch)

DataBatch(x=[6644, 88], edge_index=[2, 63920], obs_names=[6644], mask=[6644], batch=[6644], ptr=[3])


In [62]:
print(transformer_out.shape, dec_out.shape)

torch.Size([2001, 2, 16]) torch.Size([4000, 88])


In [73]:
transformer_out_list, dec_out_list = [], []
for batch_idx in np.unique(batch.batch):
    transformer_out_list.append(transformer_out[(batch.batch == batch_idx)])
    dec_out_list.append(dec_put[batch.batch == batch_idx])

IndexError: The shape of the mask [6644] at index 0 does not match the shape of the indexed tensor [2001, 2, 16] at index 0

### GCN

In [48]:
for i, batch in enumerate(dm.test_dataloader()):
    print(batch)
    x, z = model.evaluation(batch)

DataBatch(x=[6644, 88], edge_index=[2, 63920], obs_names=[6644], mask=[6644], batch=[6644], ptr=[3])


ValueError: too many values to unpack (expected 2)

In [ ]:
print(x.shape, z.shape)

In [ ]:
x_list, z_list = [], []
for batch_idx in np.unique(batch.batch):
    x_list.append(x[batch.batch == batch_idx])
    z_list.append(z[batch.batch == batch_idx])

In [ ]:
print(x_list[0].shape, x_list[1].shape)

In [ ]:
sample = 'slide1_B2-1'
idx = 1

sub_adata = adata[adata.obs['sample']==sample]
print(sub_adata)
sub_adata.layers['gnn_z'] = z_list[idx].detach().numpy()
sq.pl.spatial_scatter(
    sub_adata,
    color = ['SHH'],
    layer = 'norm_ftsqrt',
    cmap = 'viridis_r',
    shape= None
)
sq.pl.spatial_scatter(
    sub_adata,
    color = ['SHH'],
    layer = 'gnn_z',
    cmap = 'viridis_r',
    shape= None
)